# Best encoding settings

The aim here is to train a model on existing data in order to estimate the FFmpeg parameters that optimise any arbitrary criterion.

In [1]:
import subprocess
import tempfile

import cutcutcodec

from mendevi.encode import get_transcode_cmd
from mendevi.models import Solver
from mendevi.models.nn import EnergyNN, QualityNN

In [2]:
dataset = "scatter.db"  # the dataset name, automatically downloaded
val = 0.2  # keep 80% of data for training, 20% for validation
video = cutcutcodec.utils.get_project_root() / "media" / "video" / "intro.webm"

## Train the model

Before making predictions, you first need to train a model.

### Quality model

Quality does not depend on the machine. We therefore produce just one large, unique model.

In [3]:
# declaration
quality_model = QualityNN(
    output_labels=["log_rate", "psnr", "ssim"],
    weights="quality",  # optional, for checkpoint (in ~/.cache/mendevi)
    niter=10000,  # number of training iterations, default 10000
    learning_rate=1e-5,  # adam gradient decay learning rate, default 1e-4
    total=50_000,  # number of model parameters, default based on number of observations
)

Load model from /home/robin/.cache/mendevi/quality.pk.xz...
█ 11.08 ms


In [4]:
# train (on cpu encoders only)
if not quality_model.is_fit():
    quality_model.fit(
        dataset,
        table="t_enc_encode",  # optional, help to find the good SQL request
        select=f"uniform(vid_hash) >= {val} and encoder_family == 'cpu'",
    )

Fit 'Quality estimation based on a neuronal network'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster ...
█ █ this cluster contains 17853 observations
█ █ Save model into /home/robin/.cache/mendevi/quality_29_3_50000.pk.xz...                                            
█ █ █ 57.01 ms
█ █ 12 min 23 s
█ 13 min 3 s
Save model into /home/robin/.cache/mendevi/quality.pk.xz...
█ 83.52 ms


In [5]:
# validation
quality_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val} and encoder_family == 'cpu'")

Validate 'Quality estimation based on a neuronal network'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster ...
█ █ this cluster contains 4551 observations
█ █ 83.76 ms
█ 25.96 s
MAPE of log_rate: 1.37%
RMS  of log_rate: 0.119
MAPE of psnr: 2.93%
RMS  of psnr: 1.894
MAPE of ssim: 1.61%
RMS  of ssim: 0.02424
MAPE of rate: 20.91%
RMS  of rate: 2.865e+07


### Energy model

Energy depends ont the machine. We therefore produce several models, one per hostname.

In [6]:
energy_model = EnergyNN(
    weights="energy",  # optional, for checkpoint (in ~/.cache/mendevi)
    niter=4000,  # number of training iterations, default 10000
    learning_rate=1e-4,  # adam gradient decay learning rate, default 1e-4
    total=2000,  # number of model parameters, default based on number of observations
)

In [7]:
if not energy_model.is_fit():
    energy_model.fit(
        dataset,
        table="t_enc_encode",  # optional, help to find the good SQL request
        select=f"uniform(vid_hash) >= {val} encoder_family == 'cpu'",
    )

Fit 'Encoding energy estimation based on a neuronal network'...
█ get SQL extractor
█ SyntaxError l622 (the argument "not (uniform(vid_hash) >= 0.2 encoder_family == 'cpu')" is not a valid python expression)


SyntaxError: the argument "not (uniform(vid_hash) >= 0.2 encoder_family == 'cpu')" is not a valid python expression (<string>)

In [ ]:
# validation
energy_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val} encoder_family == 'cpu'")

## Use the model to perform predictions

The idea is to run a huge number of predictions using the trained model in order to explore a very large range of scenarios.

In [4]:
# example best compression for 35 dB

def best_dbrate(**kwd: dict[str]) -> float:
    """Loss function of debit rate distortion around 35 dB."""
    db_loss = 0.0 if 34 < kwd["psnr"] < 36 else ((kwd["psnr"]-35)**2)-1
    rate_loss = kwd["log_rate"]
    return db_loss + 1e-2*rate_loss

solver = Solver(quality_model, best_dbrate)
tested_combinations, loss, cmd = solver.solve(
    video,
    encoder=["libx264", "libx265", "libaom-av1", "libsvtav1", "libvvenc"],
    return_cmd=True,
)

Solve...


{'effort': ('fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fast', 'fas

█ Predict 'Quality estimation based on a neuronal network'...
█ █ check args
█ █ vectorize args
█ █ predict
█ █ Predict for cluster ...
█ █ █ this cluster contains 990 observations
█ █ █ 15.39 ms
█ █ 60.15 ms
█ ValueError l181 (zip() argument 4 is shorter than arguments 1-3)


ValueError: zip() argument 4 is shorter than arguments 1-3

In [ ]:
# perform the conversion
out_video = pathlib.Path(tempfile.gettempdir()) / "video_test.mp4"
cmd.output = [*output[:-1], str(out_video)]
output, activity = cmd.run()

In [ ]:
# final verification
subprocess.run(
    ["mendevi", "probe", str(out_video), "-r", str(video), "-d", str(out_video.with_suffix(".db")],
    check=True,
)